# 감성분석(Sentiment Analysis)

- 텍스트(리뷰, 댓글, 기사 등)에 담긴 **긍정/부정/중립** 같은 전반적 태도(Sentiment)를 분류하는 분석
- 핵심은 **텍스트 전체의 긍·부정 경향을 파악하는 것**

- [참고] 감정 분석 (Emotion Analysis)
    - 텍스트 안에서 구체적인 **감정(emotion)** 을 분류하는 분석. (예: 행복, 분노, 슬픔, 두려움, 놀람, 혐오 등)
    - 단순 긍·부정이 아니라 세분화된 감정 상태를 구분하는 것

- 감성분석은 난이도에 따라 단순한 긍/부정에서 유형별 분류까지 나아감
- 복잡한 분류일 경우 머신러닝/딥러닝의 알고리즘 또는 정교한 어휘사전이 필요함


# 1.라이브러리 불러오기

In [ ]:
import pandas as pd
import re
from konlpy.tag import Okt

# 2.데이터 셋 불러오기

- 네이버 영화 평점 데이터 셋
    - https://github.com/e9t/nsmc  

In [ ]:
train_df = 


In [ ]:
train_df.shape

In [ ]:
train_df['label'].value_counts()

In [ ]:
train_df.info()

# 3.데이터 전처리

In [ ]:
# 댓글 결측치를 빈칸으로 처리
train_df = train_df.fillna(' ')

In [ ]:
train_df.isnull().sum()

In [ ]:
result = []
for temp in train_df['document']:
    kor_str = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', temp)
    result.append(kor_str)

In [ ]:
train_df['document'] = train_df['document'].apply(lambda x : re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', x) )
train_df

In [ ]:
test_df = pd.read_csv('ratings_test.txt', sep='\t')
test_df = test_df.fillna(' ')
test_df['document'] = test_df['document'].apply(lambda x : re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', x))

In [ ]:
train_df.drop('id', axis=1, inplace=True)
test_df.drop('id', axis=1, inplace=True)

In [ ]:
# 형태소 분석
okt = Okt()

def okt_tokenizer(text):
    token_kor = okt.morphs(text)
    return token_kor

# 4.TF-IDF
## 4.1. TF-IDF란?
- TF-IDF는 문서에서 **어떤 단어가 중요한 단어인지 수치로 표현하는 방법** 
- 텍스트를 머신러닝 모델에 넣기 위해 단어를 숫자로 바꿀 때 자주 사용
- 단순히 많이 나온 단어를 중요하다고 보는 것이 아니라, **특정 문서에서 자주 나오면서 전체 문서에서는 너무 흔하지 않은 단어** 를 중요하게 본다.
- `TfidfVectorizer`로 텍스트를 수치형 데이터로 변환

## 4.2. 왜 TF-IDF가 필요한가?

- 단어 빈도만 사용하면 문제가 생긴다.
- 예를 들어 리뷰 데이터에서 `영화`, `정말`, `너무`, `그리고` 같은 단어는 자주 등장할 수 있다.
- 하지만 이런 단어들은 대부분의 문서에 널리 등장하므로 특정 문서의 특징을 잘 설명하지 못한다.
- 반대로 `감동`, `지루`, `연기`, `스토리`, `최악`, `명작` 같은 단어는 특정 리뷰의 성격을 더 잘 설명할 수 있다.
- TF-IDF는 이런 단어에 더 높은 점수를 부여한다.
---

## 4.3. [참고]TF-IDF의 핵심 아이디어

- TF-IDF는 두 값을 곱해서 만든다.

$$
TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)
$$

- `t`: 단어(term)
- `d`: 문서(document)
- `TF(t, d)`: 특정 문서 안에서 단어가 얼마나 자주 등장하는지
- `IDF(t)`: 전체 문서에서 그 단어가 얼마나 희귀한지



#### 4.3.1. TF: Term Frequency

- TF는 **특정 문서 안에서 단어가 등장한 빈도**이다.
- 어떤 단어가 한 문서 안에서 많이 등장할수록 TF 값이 커진다.

가장 단순한 TF 계산식:

$$
TF(t, d) = \text{문서 } d \text{ 안에서 단어 } t \text{가 등장한 횟수}
$$

문서 길이를 고려한 TF 계산식:

$$
TF(t, d) =
\frac{
\text{문서 } d \text{ 안에서 단어 } t \text{가 등장한 횟수}
}{
\text{문서 } d \text{의 전체 단어 수}
}
$$

예:

- 문서 A: `이 영화 영화 정말 재미있다`
- 문서 A의 전체 단어 수: 5개
- `영화` 등장 횟수: 2번

$$
TF(\text{영화}, A) = \frac{2}{5} = 0.4
$$


#### 4.3.2. IDF: Inverse Document Frequency

- IDF는 **전체 문서 중에서 해당 단어가 얼마나 드문 단어인지**를 나타낸다.
- 많은 문서에 등장하는 흔한 단어는 IDF 값이 낮다.
- 적은 문서에만 등장하는 단어는 IDF 값이 높다.

기본 IDF 계산식:

$$
IDF(t) = \log \left( \frac{N}{DF(t)} \right)
$$

- `N`: 전체 문서 개수
- `DF(t)`: 단어 `t`가 등장한 문서 개수
- `log`: 값이 너무 커지는 것을 줄이기 위한 로그 함수

예:

- 전체 문서 수 `N = 1000`
- `영화`가 등장한 문서 수 `DF(영화) = 900`
- `감동`이 등장한 문서 수 `DF(감동) = 100`

$$
IDF(\text{영화}) =
\log \left( \frac{1000}{900} \right)
$$

$$
IDF(\text{감동}) =
\log \left( \frac{1000}{100} \right)
$$

- `영화`는 대부분의 문서에 등장하므로 IDF가 낮다.
- `감동`은 상대적으로 적은 문서에 등장하므로 IDF가 높다.

---


# 4.Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

In [ ]:
lr = 


In [ ]:
params = {'C': [1, 3.5, 4.5, 5.5, 10]}

grid_cv = GridSearchCV( )

In [ ]:
grid_cv.fit( , )

In [ ]:
print(grid_cv.best_params_, round(grid_cv.best_score_, 5))

In [ ]:
tfidf_matrix_test = tfidf_vect.transform(test_df['document'])

In [ ]:
best_estimator = grid_cv.best_estimator_

In [ ]:
pred = best_estimator.predict(tfidf_matrix_test)

In [ ]:
accuracy_score(test_df['label'],pred)

In [ ]:
# 사용자 입력 리뷰에 대한 감성 분석 함수
def pred_sentiment(review):
    # 1. 데이터 클리닝
    review = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣\s]', '', review)

    # 2. Tf-idf 변환

    # 3. 감성 예측

    # 4. 결과 반환(1:긍정, 0: 부정)


